In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv")
df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df=df.drop(columns="Order_ID")
df.head()

In [ ]:
# Task 2: Write your code here:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)

In [ ]:
df=df.dropna()

In [ ]:
check_missing_values(df)

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:

categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])
  label_encoders[col] = le

df

In [ ]:
oneH_encoders = {}
for col in categorical_cols:
  oneH = OneHotEncoder()
  df[col] = oneH.fit_transform(df[col])
  oneH_encoders[col] = oneH

df

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()


In [ ]:
# Task 6: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1).astype(float)
y = df['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error



In [ ]:
model= RandomForestClassifier(
      n_estimators=320,  # Number of trees
      max_depth=4
  )


In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
rmse_scores = []

for train_idx, val_idx in skf.split(X,y):
    X_fold_train, X_fold_val = X.iloc[train_idx], X.iloc[val_idx]
    y_fold_train, y_fold_val = y.iloc[train_idx], y.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))
    rmse_scores.append(np.sqrt(mean_squared_error(y_fold_val, y_fold_pred)))

mae_scores = np.array(mae_scores)
rmse_scores = np.array(rmse_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")
print(f"RMSE: ${rmse_scores.mean():,.2f}")



In [ ]:
# Task 1: Write your code here:
# Feature importance
feature_cols = ['Distance_km', 'Weather' ,	'Traffic_Level',	'Time_of_Day',	'Vehicle_Type',	'Preparation_Time_min',	'Courier_Experience_yrs']
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

plt.figure(figsize=(10, 5))
plt.hist(y_fold_pred, bins=30, edgecolor='black')
plt.title('predicted delivery time')
plt.show()

In [ ]:
# Task Bonus: Write your code here: